# Transform Constructors Data

1. Read bronze `constructors` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`constructorId` → `constructor_id`)
1. Rename columns to make them more meaningful (`name` → `constructor_name`)
1. Remove duplicate records
1. Transform values of column `nationality` to Title Case
1. Write the transformed data to silver `constructors` table


In [0]:
%run ../00_common/01_configuration

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"

In [0]:
from pyspark.sql import functions as F

#### Step 1 - Read bronze `constructors` table

In [0]:
constructors_df = spark.table(bronze_table)

#### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
constructors_dropped_df = constructors_df.drop("url")

#### Step 3 & 4 - Standardise Column Names
- Standardise column names using snake_case (`constructorId` → `constructor_id`)
- Rename columns to make them more meaningful (`name` → `constructor_name`)

In [0]:
constructors_renamed_df = (
    constructors_dropped_df
        .withColumnsRenamed({
            "constructorId": "constructor_id",
            "name": "constructor_name"
        })
)

In [0]:
display(constructors_renamed_df)

constructor_id,constructor_name,nationality,ingestion_timestamp,source_file
adams,Adams,american,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
afm,AFM,german,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
ags,AGS,french,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alfa,Alfa Romeo,swiss,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alphatauri,AlphaTauri,italian,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alpine,Alpine F1 Team,french,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alta,Alta,british,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
amon,Amon,new zealander,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
apollon,Apollon,swiss,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
arrows,Arrows,british,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json


#### Step 5 - Remove duplicate records

In [0]:
constructors_distinct_df = constructors_renamed_df.dropDuplicates(["constructor_id"])

In [0]:
display(constructors_distinct_df)

constructor_id,constructor_name,nationality,ingestion_timestamp,source_file
adams,Adams,american,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
afm,AFM,german,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
ags,AGS,french,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alfa,Alfa Romeo,swiss,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alphatauri,AlphaTauri,italian,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alpine,Alpine F1 Team,french,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alta,Alta,british,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
amon,Amon,new zealander,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
apollon,Apollon,swiss,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
arrows,Arrows,british,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json


#### Step 6 - Transform values of column `nationality` to Title Case

In [0]:
constructors_final_df = (
    constructors_distinct_df
        .withColumn('nationality', F.initcap(F.col("nationality")))
)

In [0]:
display(constructors_final_df)

constructor_id,constructor_name,nationality,ingestion_timestamp,source_file
adams,Adams,American,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
afm,AFM,German,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
ags,AGS,French,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alfa,Alfa Romeo,Swiss,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alphatauri,AlphaTauri,Italian,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alpine,Alpine F1 Team,French,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alta,Alta,British,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
amon,Amon,New Zealander,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
apollon,Apollon,Swiss,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
arrows,Arrows,British,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json


#### Step 7 - Write the transformed data to silver `constructors` table

In [0]:
(
    constructors_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))

constructor_id,constructor_name,nationality,ingestion_timestamp,source_file
adams,Adams,American,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
afm,AFM,German,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
ags,AGS,French,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alfa,Alfa Romeo,Swiss,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alphatauri,AlphaTauri,Italian,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alpine,Alpine F1 Team,French,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alta,Alta,British,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
amon,Amon,New Zealander,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
apollon,Apollon,Swiss,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
arrows,Arrows,British,2026-08-10T18:56:53.926863Z,dbfs:/Volumes/formula1/landing/files/constructors.json
